In [0]:
spark.catalog.setCurrentCatalog("databricks_project")
spark.catalog.setCurrentDatabase("silver_schema")
display(spark.catalog.currentCatalog(), ",", spark.catalog.currentDatabase())

In [0]:
'''column = "_rescued_data"

bronze_tables = ["album", "artist", "customer", "employee", "genre", "invoice_line", "invoice", "media_type", "track"]

for table in bronze_tables:
    if column in spark.read.table(f"databricks_project.bronze_schema.{table}").columns:
        table_read = spark.read.table(f"databricks_project.bronze_schema.{table}").drop(column)
    else:
        table_read = spark.read.table(f"databricks_project.bronze_schema.{table}")
    display(table_read.head(5))

'''
album_read = spark.read.table("databricks_project.bronze_schema.album").drop("_rescued_data")
artist_read = spark.read.table("databricks_project.bronze_schema.artist").drop("_rescued_data")
customer_read = spark.read.table("databricks_project.bronze_schema.customer").drop("_rescued_data")
employee_read = spark.read.table("databricks_project.bronze_schema.employee").drop("_rescued_data")
genre_read = spark.read.table("databricks_project.bronze_schema.genre").drop("_rescued_data")
invoice_line_read = spark.read.table("databricks_project.bronze_schema.invoice_line").drop("_rescued_data")  
invoice_read = spark.read.table("databricks_project.bronze_schema.invoice").drop("_rescued_data")
media_type_read = spark.read.table("databricks_project.bronze_schema.media_type").drop("_rescued_data")
track_read = spark.read.table("databricks_project.bronze_schema.track").drop("_rescued_data")


display(album_read.head(10))


In [0]:
display(album_read)
display(artist_read)
display(customer_read)
display(employee_read)
display(genre_read)
display(invoice_line_read)
display(invoice_read)
display(media_type_read)
display(track_read)

### Null Handling

In [0]:
# function to check null
from pyspark.sql.functions import col, sum as _sum

def null_check(df):
    null_count = df.select([_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
    return null_count

display(null_check(album_read))
display(null_check(artist_read))
display(null_check(customer_read))
display(null_check(employee_read))
display(null_check(genre_read))
display(null_check(invoice_line_read))
display(null_check(invoice_read))
display(null_check(media_type_read))
display(null_check(track_read))


In [0]:
# function to trim, apply title case and convert date columns to date type
from pyspark.sql import functions as F;

def basic_clean(df, string_columns, date_columns):
    
    for c,t in df.dtypes:   #trim
        if t == 'string':
            df = df.withColumn(c, F.trim(F.col(c)))
    
    if string_columns:      # title case
        for c in string_columns:
            df = df.withColumn(c, F.initcap(F.lower(F.col(c))))

    if date_columns:        # date type
        for c in date_columns:
            df = df.withColumn(c, F.col(c).cast("date"))
    
    return df



album_read = basic_clean(album_read, string_columns=['Title'], date_columns=None)
artist_read = basic_clean(artist_read, string_columns=['Name'], date_columns=None)
customer_read = basic_clean(customer_read, string_columns=['FirstName', 'LastName', 'Company', 'Address', 'City', 'State', 'Country', 'PostalCode', 'Phone', 'Fax', 'Email'], date_columns=None)
employee_read = basic_clean(employee_read, string_columns=['FirstName', 'LastName', 'Address', 'City', 'State', 'Country', 'PostalCode', 'Phone', 'Fax', 'Email'], date_columns=['BirthDate', 'HireDate'])
genre_read = basic_clean(genre_read, string_columns=['Name'], date_columns=None)
invoice_line_read = basic_clean(invoice_line_read, string_columns=None, date_columns=None)
invoice_read = basic_clean(invoice_read, string_columns=['BillingAddress', 'BillingCity', 'BillingState', 'BillingCountry', 'BillingPostalCode'], date_columns=['InvoiceDate'])
media_type_read = basic_clean(media_type_read, string_columns=['Name'], date_columns=None)
track_read = basic_clean(track_read, string_columns=['Name', 'Composer'], date_columns=None)

In [0]:
album_read = album_read.dropDuplicates(["AlbumId"])
artist_read = artist_read.dropDuplicates(["ArtistId"])
customer_read = customer_read.dropDuplicates(["CustomerId"])
employee_read = employee_read.dropDuplicates(["EmployeeId"])
genre_read = genre_read.dropDuplicates(["GenreId"])
#invoice_line_read = invoice_line_read.dropDuplicates(["InvoiceLineId"])
invoice_read = invoice_read.dropDuplicates(["InvoiceId"])
media_type_read = media_type_read.dropDuplicates(["MediaTypeId"])
track_read = track_read.dropDuplicates(["TrackId"])

In [0]:
data = [("SP", "São José dos Campos", "Brazil"),
    ("BW", "Stuttgart", "Germany"),  
    ("QC", "Montréal", "Canada"),
    ("OSL", "Oslo", "Norway"),
    ("PR", "Prague", "Czech Republic"),
    ("Vienne", "Vienne", "Austria"), 
    ("BRU", "Brussels", "Belgium"),
    ("DK", "Copenhagen", "Denmark"),
    ("SP", "São Paulo", "Brazil"),
    ("RJ", "Rio de Janeiro", "Brazil"),
    ("DF", "Brasília", "Brazil"),
    ("AB", "Edmonton", "Canada"),
    ("BC", "Vancouver", "Canada"),
    ("CA", "Mountain View", "USA"),
    ("WA", "Redmond", "USA"),
    ("NY", "New York", "USA"),
    ("CA", "Cupertino", "USA"),
    ("NV", "Reno", "USA"),
    ("FL", "Orlando", "USA"),
    ("MA", "Boston", "USA"),
    ("IL", "Chicago", "USA"),
    ("WI", "Madison", "USA"),
    ("TX", "Fort Worth", "USA"),
    ("AZ", "Tucson", "USA"),
    ("UT", "Salt Lake City", "USA"),
    ("ON", "Toronto", "Canada"),
    ("ON", "Ottawa", "Canada"),
    ("NS", "Halifax", "Canada"),
    ("MB", "Winnipeg", "Canada"),
    ("NT", "Yellowknife", "Canada"),
    ("Lisbon", "Lisbon", "Portugal"),  
    ("Porto", "Porto", "Portugal"),   
    ("BE", "Berlin", "Germany"),
    ("HE", "Frankfurt", "Germany"), 
    ("IDF", "Paris", "France"),   
    ("ARA", "Lyon", "France"),    
    ("NAQ", "Bordeaux", "France"),
    ("BFC", "Dijon", "France"),   
    ("UUS", "Helsinki", "Finland"),
    ("BU", "Budapest", "Hungary"),
    ("DBL", "Dublin", "Ireland"),
    ("RM", "Rome", "Italy"),
    ("NH", "Amsterdam", "Netherlands"),  
    ("MZ", "Warsaw", "Poland"),   
    ("MD", "Madrid", "Spain"),
    ("AB", "Stockholm", "Sweden"), 
    ("ENG", "London", "United Kingdom"),
    ("SCT", "Edinburgh", "United Kingdom"),
    ("NSW", "Sidney", "Australia"),
    ("BA", "Buenos Aires", "Argentina"),
    ("RM", "Santiago", "Chile"),  
    ("DL", "Delhi", "India"),
    ("KA", "Bangalore", "India")
    ]
columns = ["States_state", "States_city", "States_country"]
States = spark.createDataFrame(data, columns)

display(States)

In [0]:
# imputing State columns in customer and Invoice df's

customer_read_new = customer_read.alias('c').\
    join(States.alias('s'), (F.lower(F.col('c.City')) == F.lower(F.col('s.States_city'))) & (F.lower(F.col('c.Country')) == F.lower(F.col('s.States_country'))), 'left').\
    withColumn('States', F.when(F.col('c.State').isNull(), F.col('s.States_state')).otherwise(F.col('c.State'))).\
    drop("State","States_city","States_state","States_country")

invoice_read_new = invoice_read.alias("i").\
    join(States.alias('s'), (F.lower(F.col('i.BillingCity')) == F.lower(F.col('s.States_city'))) & (F.lower(F.col('i.BillingCountry')) == F.lower(F.col('s.States_country'))), 'left').\
    withColumn('States', F.when(F.col('i.BillingState').isNull(), F.col('s.States_state')).otherwise(F.col('i.BillingState'))).\
    drop("BillingState","States_city","States_state","States_country")

#customer_read_new = customer_read_new.dropna(how = 'any', subset=["PostalCode","Phone","States"]).drop("Fax","Company")
#invoice_read_new = invoice_read_new.dropna(how='any', subset=["BillingPostalCode", "States"]).drop()

In [0]:
employee_read_new = employee_read.withColumn("ReportsTo", F.when(F.col("EmployeeId")==1, 1).otherwise(F.col("ReportsTo")))
track_read_new = track_read.fillna('Unknown', ['Composer'])

### Fact and dim tables

In [0]:
from pyspark.sql.window import Window

dim_artist = artist_read.select("ArtistId", "Name")

dim_album = album_read.select("AlbumId", "Title", "ArtistId")

dim_genre = genre_read.select("GenreId", "Name")

dim_mediatype = media_type_read.select("MediaTypeId", "Name")

w_track = Window.orderBy("TrackId")

dim_track = track_read_new.alias("t").\
    withColumn("Track_sk", F.row_number().over(w_track)).\
        join(dim_album.alias("a"), F.col("t.AlbumId")==F.col("a.AlbumId"), "left")\
    .join(dim_artist.alias("ar"), F.col("a.ArtistId")==F.col("ar.ArtistId"), "left")\
    .join(dim_mediatype.alias("m"), F.col("t.MediaTypeId")==F.col("m.MediaTypeId"), "left")\
    .join(dim_genre.alias("g"), F.col("t.GenreId")==F.col("g.GenreId"), "left")\
    .select("Track_sk",
                "t.TrackId", 
                "t.Name", 
                "t.AlbumId",
                "t.MediaTypeId", 
                "t.GenreId",
                F.col("a.Title").alias("Album"),
                F.col("ar.Name").alias("Artist"), 
                F.col("g.Name").alias("Genre"),
                F.col("m.Name").alias("MediaType"),
                "t.Composer",
                "t.Milliseconds", 
                "t.Bytes")

w_customer = Window.orderBy("CustomerId")

dim_customer = customer_read_new.withColumn("Customer_sk", F.row_number().over(w_customer)).select("Customer_sk","CustomerId","FirstName","LastName","Address","City","Country","PostalCode","Phone","Email","SupportRepId","States")

w_employee = Window.orderBy("EmployeeId")

dim_employee = employee_read_new.withColumn("Employee_sk", F.row_number().over(w_employee)).select("Employee_sk","EmployeeId","FirstName","LastName","Title","ReportsTo","BirthDate","HireDate","Address","City","State","Country","PostalCode")

w_invoice = Window.orderBy("InvoiceId")

dim_invoice = invoice_read_new.withColumn("Invoice_sk", F.row_number().over(w_invoice)).select("Invoice_sk","InvoiceId","CustomerId","InvoiceDate","BillingAddress","BillingCity","BillingCountry","BillingPostalCode","States")

In [0]:
fact_inv_line = (
    invoice_line_read.alias("il")
    .join(dim_track.alias("t"), F.col("il.TrackId") == F.col("t.TrackId"), "left")
    .join(dim_invoice.alias("i"), F.col("il.InvoiceId") == F.col("i.InvoiceId"), "left")
    .join(dim_customer.alias("c"), F.col("i.CustomerId") == F.col("c.CustomerId"), "left")
    .join(dim_employee.alias("e"), F.col("c.SupportRepId") == F.col("e.EmployeeId"), "left")
    .withColumn("Trk_sk", F.when(F.col("Track_sk").isNotNull(), F.col("Track_sk")).otherwise(F.lit(0)))
    .withColumn("Inv_sk", F.when(F.col("Invoice_sk").isNotNull(), F.col("Invoice_sk")).otherwise(F.lit(0)))
    .withColumn("Cust_sk", F.when(F.col("Customer_sk").isNotNull(), F.col("Customer_sk")).otherwise(F.lit(0)))
    .withColumn("Emp_sk", F.when(F.col("Employee_sk").isNotNull(), F.col("Employee_sk")).otherwise(F.lit(0)))
    .select(
        F.col("il.InvoiceLineId"),
        F.col("Inv_sk"),
        F.col("Trk_sk"),
        F.col("Cust_sk"),
        F.col("Emp_sk"), 
        F.col("i.InvoiceDate"),         
        F.col("il.UnitPrice"),
        F.col("il.Quantity"),
    )
)


In [0]:
dim_album.write.mode("overwrite").saveAsTable("databricks_project.silver_schema.dim_album")
dim_artist.write.mode("overwrite").saveAsTable("databricks_project.silver_schema.dim_artist")
dim_customer.write.mode("overwrite").saveAsTable("databricks_project.silver_schema.dim_customer")
dim_employee.write.mode("overwrite").saveAsTable("databricks_project.silver_schema.dim_employee")
dim_genre.write.mode("overwrite").saveAsTable("databricks_project.silver_schema.dim_genre")
dim_invoice.write.mode("overwrite").saveAsTable("databricks_project.silver_schema.dim_invoice")
dim_mediatype.write.mode("overwrite").saveAsTable("databricks_project.silver_schema.dim_mediatype")
dim_track.write.mode("overwrite").saveAsTable("databricks_project.silver_schema.dim_track")
fact_inv_line.write.mode("overwrite").saveAsTable("databricks_project.silver_schema.fact_inv_line")

In [0]:
display(dim_album)
display(dim_artist)
display(dim_customer)
display(dim_employee)
display(dim_genre)
display(dim_invoice)
display(dim_mediatype)
display(dim_track)
display(fact_inv_line)

In [0]:
%sql

ALTER TABLE dim_employee ALTER COLUMN Employee_sk SET NOT NULL;

ALTER TABLE dim_employee ADD CONSTRAINT employee_pk PRIMARY KEY (Employee_sk);
------------------

ALTER TABLE dim_customer ALTER COLUMN Customer_sk SET NOT NULL;

ALTER TABLE dim_customer ADD CONSTRAINT customer_pk PRIMARY KEY (Customer_sk);

------------------

ALTER TABLE dim_invoice ALTER COLUMN Invoice_sk SET NOT NULL;

ALTER TABLE dim_invoice ADD CONSTRAINT invoice_pk PRIMARY KEY (Invoice_sk);

----

ALTER TABLE dim_track ALTER COLUMN Track_sk SET NOT NULL;

ALTER TABLE dim_track ADD CONSTRAINT track_pk PRIMARY KEY (Track_sk);
-----------

ALTER TABLE fact_inv_line ALTER COLUMN InvoiceLineId SET NOT NULL;

ALTER TABLE fact_inv_line ADD CONSTRAINT invline_pk PRIMARY KEY (InvoiceLineId);

ALTER TABLE fact_inv_line ADD CONSTRAINT invoice_fk FOREIGN KEY (Inv_sk) REFERENCES dim_invoice(Invoice_sk);

ALTER TABLE fact_inv_line ADD CONSTRAINT track_fk FOREIGN KEY (Trk_sk) REFERENCES dim_track(Track_sk);

ALTER TABLE fact_inv_line ADD CONSTRAINT customer_fk FOREIGN KEY(Cust_sk) REFERENCES dim_customer(Customer_sk);

ALTER TABLE fact_inv_line ADD CONSTRAINT employee_fk FOREIGN KEY (Emp_sk) REFERENCES dim_employee(Employee_sk);
